# Cải tiến NPR trên Kaggle GPU — 2 thí nghiệm + đánh giá 5 benchmark

| Thí nghiệm | Nội dung | Nhắm vào |
|---|---|---|
| **npr_aug** | NPR + augmentation Blur/JPEG (p=0.5, công thức CNNDetection) | Báo động giả trên ảnh thật (GANGen Real-Acc 31%), ảnh nén từ web |
| **npr_adaptive_aug** | AdaptiveNPR (residual đa tỉ lệ có học) + augmentation | Dấu vết upsampling đa dạng (Diffusion, GAN lạ) |

**Chuẩn bị:** Accelerator = GPU T4/P100 · Add Input dataset benchmark · Internet ON · Run All.
Thời gian: ~3-4h/thí nghiệm + ~1.5h đánh giá. Nếu sợ hết giờ, đặt `EXPERIMENTS = ['aug']` chạy trước, phiên sau chạy `['adaptive']`.

In [ ]:
EXPERIMENTS = ['aug', 'adaptive']   # sửa thành ['aug'] hoặc ['adaptive'] nếu muốn tách phiên
EPOCHS = 8

!rm -rf Deepfake-Detect && git clone -q --depth 1 https://github.com/KimThanhTran/Deepfake-Detect.git
%cd Deepfake-Detect
!pip -q install scikit-learn tqdm 2>/dev/null
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

import glob, os
hits = sum((glob.glob(f'/kaggle/input/{d}/ForenSynths/ForenSynths/train') for d in ('*', '*/*', '*/*/*')), [])
assert hits, 'Không tìm thấy ForenSynths/ForenSynths/train trong /kaggle/input'
DATA = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
print('DATA =', DATA)

In [ ]:
# Train các thí nghiệm (setup gốc NPR: Adam lr 2e-4 batch 32 + cải tiến tương ứng)
runs = {}
if 'aug' in EXPERIMENTS:
    runs['npr_aug'] = ''
if 'adaptive' in EXPERIMENTS:
    runs['npr_adaptive_aug'] = '--adaptive_npr'

for name, extra in runs.items():
    !python train.py --name {name} \
        --dataroot {DATA}/ForenSynths/ForenSynths --classes car,cat,chair,horse \
        --batch_size 32 --lr 0.0002 --niter {EPOCHS} --num_threads 4 \
        --blur_prob 0.5 --blur_sig 0.0,3.0 --jpg_prob 0.5 --jpg_method cv2,pil --jpg_qual 30,100 \
        --loss_freq 2000 --skip_bench_eval {extra}

In [ ]:
# Thu checkpoint (thư mục train có timestamp trong tên)
import glob, shutil, os
os.makedirs('/kaggle/working/ckpt', exist_ok=True)
ckpts = {}
for name in runs:
    c = sorted(glob.glob(f'checkpoints/{name}*/model_epoch_last.pth'))[-1]
    dst = f'/kaggle/working/ckpt/{name}.pth'
    shutil.copy(c, dst); ckpts[name] = dst
    print(name, '->', dst)

In [ ]:
# Đánh giá trên 5 benchmark (protocol NPR: resize 256, không crop)
import os
sets = {
    'ForenSynths-test':    f'{DATA}/ForenSynths/ForenSynths/test',
    'GANGen-Detection':    f'{DATA}/GANGen-Detection/GANGen-Detection',
    'UniversalFakeDetect': f'{DATA}/UniversalFakeDetect/UniversalFakeDetect',
    'DiffusionForensics':  f'{DATA}/DiffusionForensics/DiffusionForensics',
    'Diffusion1kStep':     f'{DATA}/Diffusion1kStep/Diffusion1kStep',
}
for name, ck in ckpts.items():
    for ds, root in sets.items():
        if not os.path.isdir(root):
            print('[skip]', ds); continue
        !python tools/eval_report.py --model_path {ck} --dataroot {root} \
            --out_dir /kaggle/working/results/{name}_{ds} \
            --label "{name} - {ds}" --batch_size 64 --num_workers 4

In [ ]:
# Tổng hợp
!python tools/augment_metrics.py /kaggle/working/results
import pandas as pd
print(pd.read_csv('/kaggle/working/results/summary_all_runs.csv').to_string(index=False))